In [ ]:
from __future__ import annotations

from app.utils.helpers import clamp, utc_now,record_activity


class MasteryService:
    """
    Maintains persistent concept-level learner mastery.

    Mastery uses repeated evidence instead of replacing the
    learner's previous state with every new assessment.

    The current evidence receives more weight than old evidence,
    while historical performance is preserved.
    """

    EVIDENCE_WEIGHTS = {
        "assessment": 0.35,
        "quiz_answer": 0.35,
        "open_ended": 0.40,
        "tutor_check": 0.25,
        "practice": 0.30,
    }

    def __init__(self, database):
        self.database = database

    # ========================================================
    # UPDATE
    # ========================================================

    def update_from_evidence(
        self,
        user_id: str,
        project_id: str,
        concept_id: str,
        evidence_score: float,
        evidence_type: str = "assessment",
        evidence_description: str | None = None,
        evidence_confidence: float | None = None,
    ):

        if not user_id:
            raise ValueError(
                "user_id is required."
            )

        if not project_id:
            raise ValueError(
                "project_id is required."
            )

        if not concept_id:
            raise ValueError(
                "concept_id is required."
            )

        collection = self.database.collection(
            "mastery"
        )

        evidence_score = clamp(
            float(evidence_score),
            0.0,
            1.0,
        )

        now = utc_now()

        existing = collection.find_one(
            {
                "user_id": user_id,
                "project_id": project_id,
                "concept_id": concept_id,
            },
            {
                "_id": 0,
            },
        )

        weight = self.EVIDENCE_WEIGHTS.get(
            evidence_type,
            0.35,
        )

        # Allow stronger/weaker evidence to affect
        # how much the new observation influences mastery.
        if evidence_confidence is not None:

            confidence = clamp(
                float(evidence_confidence),
                0.0,
                1.0,
            )

            weight *= (
                0.75
                + 0.50 * confidence
            )

            weight = clamp(
                weight,
                0.15,
                0.60,
            )

        # ====================================================
        # FIRST EVIDENCE
        # ====================================================

        if existing is None:

            score = evidence_score
            confidence = (
                0.25
                if evidence_confidence is None
                else clamp(
                    float(evidence_confidence),
                    0.0,
                    1.0,
                )
            )

            trend = self._calculate_trend(
                previous_score=None,
                new_score=score,
            )

            recent_evidence = []

            if evidence_description:
                recent_evidence.append(
                    f"{evidence_type}: "
                    f"{evidence_description}"
                )

            document = {
                "id": (
                    f"{user_id}:"
                    f"{project_id}:"
                    f"{concept_id}"
                ),
                "user_id": user_id,
                "project_id": project_id,
                "concept_id": concept_id,
                "score": score,
                "confidence": confidence,
                "trend": trend,
                "assessment_count": 1,
                "last_assessed_at": now,
                "recent_evidence": recent_evidence[-5:],
                "created_at": now,
                "updated_at": now,
            }

            collection.insert_one(
                document
            )

            record_activity(
                self.database,
                user_id=user_id,
                project_id=project_id,
                event_type="MASTERY_UPDATED",
                description=f"Updated mastery for {concept_id}",
                entity_type="mastery",
                entity_id=document["id"],
                metadata={
                    "concept_id": concept_id,
                    "score": score,
                    "trend": trend,
                },
            )

            return {
                key: value
                for key, value in document.items()
                if key != "_id"
            }

        # ====================================================
        # EXISTING MASTERY
        # ====================================================

        old_score = clamp(
            float(
                existing.get(
                    "score",
                    0.0,
                )
            ),
            0.0,
            1.0,
        )

        old_confidence = clamp(
            float(
                existing.get(
                    "confidence",
                    0.0,
                )
            ),
            0.0,
            1.0,
        )

        old_count = max(
            int(
                existing.get(
                    "assessment_count",
                    0,
                )
            ),
            0,
        )

        score = (
            old_score * (1.0 - weight)
            + evidence_score * weight
        )

        score = clamp(
            score,
            0.0,
            1.0,
        )

        # Confidence increases as more independent evidence
        # accumulates, but never reaches certainty automatically.
        confidence_gain = (
            0.08
            * (
                1.0
                - old_confidence
            )
        )

        if evidence_confidence is not None:

            confidence_gain *= (
                0.50
                + 0.50
                * clamp(
                    float(evidence_confidence),
                    0.0,
                    1.0,
                )
            )

        new_confidence = clamp(
            old_confidence
            + confidence_gain,
            0.0,
            0.95,
        )

        trend = self._calculate_trend(
            previous_score=old_score,
            new_score=score,
        )

        recent_evidence = list(
            existing.get(
                "recent_evidence",
                [],
            )
        )

        if evidence_description:

            recent_evidence.append(
                f"{evidence_type}: "
                f"{evidence_description}"
            )

        recent_evidence = (
            recent_evidence[-5:]
        )

        document = {
            "score": score,
            "confidence": new_confidence,
            "trend": trend,
            "assessment_count": old_count + 1,
            "last_assessed_at": now,
            "recent_evidence": recent_evidence,
            "updated_at": now,
        }

        collection.update_one(
            {
                "user_id": user_id,
                "project_id": project_id,
                "concept_id": concept_id,
            },
            {
                "$set": document,
            },
        )

        record_activity(
            self.database,
            user_id=user_id,
            project_id=project_id,
            event_type="MASTERY_UPDATED",
            description=f"Updated mastery for {concept_id}",
            entity_type="mastery",
            entity_id=f"{user_id}:{project_id}:{concept_id}",
            metadata={
                "concept_id": concept_id,
                "score": score,
                "trend": trend,
            },
        )

        return {
            **existing,
            **document,
        }

    # ========================================================
    # TREND
    # ========================================================

    @staticmethod
    def _calculate_trend(
        previous_score: float | None,
        new_score: float,
    ) -> str:

        if new_score < 0.50:
            return "needs_attention"

        if previous_score is None:
            if new_score >= 0.70:
                return "improving"

            return "stable"

        delta = (
            new_score
            - previous_score
        )

        if delta >= 0.05:
            return "improving"

        if new_score < 0.50:
            return "needs_attention"

        return "stable"

    # ========================================================
    # READ
    # ========================================================

    def get_project_mastery(
        self,
        user_id: str,
        project_id: str,
    ) -> list[dict]:

        return list(
            self.database.collection(
                "mastery"
            ).find(
                {
                    "user_id": user_id,
                    "project_id": project_id,
                },
                {
                    "_id": 0,
                },
            ).sort(
                "score",
                1,
            )
        )

    def get_concept_mastery(
        self,
        user_id: str,
        project_id: str,
        concept_id: str,
    ) -> dict | None:

        return self.database.collection(
            "mastery"
        ).find_one(
            {
                "user_id": user_id,
                "project_id": project_id,
                "concept_id": concept_id,
            },
            {
                "_id": 0,
            },
        )
